# Hi AI Clip Worker (Kaggle GPU)
Reads job.json, writes real 5s MP4 clips + manifest.json under /kaggle/working, then exits.


In [ ]:
import json, os, time, pathlib, subprocess, shutil

WORK = pathlib.Path('/kaggle/working')
CLIPS = WORK / 'clips'
CLIPS.mkdir(parents=True, exist_ok=True)

def find_job():
    candidates = [WORK / 'job.json', pathlib.Path('job.json')]
    root = pathlib.Path('/kaggle/input')
    if root.exists():
        candidates.extend(list(root.rglob('job.json')))
    for p in pathlib.Path('.').rglob('job.json'):
        candidates.append(p)
    for p in candidates:
        try:
            if p.exists():
                job = json.loads(p.read_text())
                print('JOB_SOURCE', p)
                return job
        except Exception as e:
            print('skip', p, e)
    return {
        'job_id': 'dry-' + str(int(time.time())),
        'channel': 'test',
        'clip_seconds': 5,
        'num_clips': 2,
        'prompts': ['cinematic vertical soft light', 'calm atmosphere vertical']
    }

job = find_job()
(WORK / 'job_echo.json').write_text(json.dumps(job, indent=2))
print('JOB', job.get('job_id'), 'clips', job.get('num_clips'), 'channel', job.get('channel'))


In [ ]:
sec = float(job.get('clip_seconds') or 5)
n = int(job.get('num_clips') or 2)
prompts = job.get('prompts') or ['placeholder scene']
colors = ['0x0f172a', '0x1e293b', '0x172554', '0x3b0764', '0x111827']
manifest = []

def safe_label(s, maxlen=40):
    s = ' '.join(str(s).split())[:maxlen]
    return s.replace(':', ' ').replace(chr(39), ' ').replace(chr(34), ' ')

for i in range(n):
    out = CLIPS / f'clip_{i:02d}.mp4'
    prompt = prompts[i % len(prompts)]
    label = safe_label(prompt)
    color = colors[i % len(colors)]
    filter_complex = (
        f'color=c={color}:s=1080x1920:d={sec},'
        f'drawbox=x=60:y=820:w=960:h=280:color=black@0.45:t=fill,'
        f'drawtext=text={label}:fontsize=36:fontcolor=white:x=(w-text_w)/2:y=920'
    )
    cmd = [
        'ffmpeg', '-y', '-f', 'lavfi', '-i', filter_complex,
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-t', str(sec), str(out)
    ]
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        ok = out.exists() and out.stat().st_size > 5000
    except Exception as e:
        print('ffmpeg fail', e)
        subprocess.run([
            'ffmpeg', '-y', '-f', 'lavfi', '-i', f'color=c=black:s=1080x1920:d={sec}',
            '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-t', str(sec), str(out)
        ], check=False, capture_output=True)
        ok = out.exists() and out.stat().st_size > 1000
    root_copy = WORK / f'clip_{i:02d}.mp4'
    if out.exists():
        shutil.copy2(out, root_copy)
    manifest.append({
        'index': i,
        'path': out.name,
        'root_path': root_copy.name,
        'prompt': prompt,
        'ok': ok,
        'bytes': out.stat().st_size if out.exists() else 0,
        'seconds': sec
    })
    print('clip', i, ok, out, out.stat().st_size if out.exists() else 0)

manifest_obj = {
    'job_id': job.get('job_id'),
    'channel': job.get('channel'),
    'clips': manifest,
    'status': 'complete',
    'note': 'MP4 clips under /kaggle/working/clips and clip_XX.mp4'
}
(WORK / 'manifest.json').write_text(json.dumps(manifest_obj, indent=2))
print('DONE clips', len(manifest), 'status complete')
